In [2]:
from pathlib import Path
import json
import polars as pl

In [ ]:
"""bike_riders_path = Path(
    "/home/najla/dev/najla-msc/data/raw/THATS/bike_riders/bike_share_ridership.parquet"
)"""
bike_riders_path = Path(
    "/home/najla/dev/najla-msc/bikeshare/data/raw/bike_ridership/ridership/bike_share_ridership.parquet"
)


"""grouped_output_path = Path(
    "/home/najla/dev/najla-msc/bikeshare/data/processed/bike_ridership/bike_riders_grouped_daily_5yrs_12pm.parquet"
)

# same data but in the new folder
grouped_output_path = Path(
    "/home/najla/dev/najla-msc/bikeshare/data/processed/bike_ridership/bike_riders_grouped_daily.parquet"
)"""
grouped_output_path = Path(
    "/home/najla/dev/najla-msc/bikeshare/data/processed/bike_ridership/bike_riders_grouped_daily_5yrs_12pm.parquet"
)

bike_riders_lazy = pl.scan_parquet(bike_riders_path)

bike_riders_lazy.collect_schema()

Schema([('trip_id', Int64),
        ('trip_duration', String),
        ('start_station_id', Float64),
        ('start_time', String),
        ('start_station_name', String),
        ('end_station_id', Float64),
        ('end_time', String),
        ('end_station_name', String),
        ('bike_id', Int64),
        ('user_type', String),
        ('source_file', String),
        ('bike_model', String)])

In [4]:
# prev.
grouped_lazy = (
    bike_riders_lazy
    .select(
        "trip_id",
        "trip_duration",
        "end_time",
        "end_station_name",
        "start_station_name",
    )
    .with_columns(
        pl.col("end_time")
        .cast(pl.Utf8)
        .str.strptime(pl.Datetime, format=None, strict=False)
        .dt.date()
        .alias("end_date"),

        pl.col("trip_duration")
        .cast(pl.Utf8)
        .str.replace_all(",", "")
        .cast(pl.Float64, strict=False)
        .alias("trip_duration_num"),
    )
    .filter(
        pl.col("end_date").is_not_null()
        & pl.col("end_station_name").is_not_null()
        & pl.col("start_station_name").is_not_null()
    )
    .group_by(
        [
            "end_date",
            "end_station_name",
            "start_station_name",
        ]
    )
    .agg(
        pl.col("trip_duration_num").mean().alias("avg_trip_duration"),
        pl.col("trip_id").count().alias("trip_count"),
    )
)

In [6]:
# fixed + filter time before 12 pm
grouped_lazy = (
    bike_riders_lazy
    .select(
        "trip_id",
        "trip_duration",
        "start_time",
        "end_time",
        "end_station_name",
        "start_station_name",
    )
    .with_columns(
        pl.coalesce(
            pl.col("start_time").cast(pl.Utf8).str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False),
            pl.col("start_time").cast(pl.Utf8).str.strptime(pl.Datetime, format="%m/%d/%Y %H:%M", strict=False),
        ).alias("start_time_dt"),

        pl.coalesce(
            pl.col("end_time").cast(pl.Utf8).str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False),
            pl.col("end_time").cast(pl.Utf8).str.strptime(pl.Datetime, format="%m/%d/%Y %H:%M", strict=False),
        ).alias("end_time_dt"),

        pl.col("trip_duration")
        .cast(pl.Utf8)
        .str.replace_all(",", "")
        .cast(pl.Float64, strict=False)
        .alias("trip_duration_num"),
    )
    .with_columns(
        pl.col("end_time_dt").dt.date().alias("end_date")
    )
    .filter(
        pl.col("end_date").is_not_null()
        & pl.col("start_time_dt").is_not_null()
        & pl.col("end_time_dt").is_not_null()
        & (pl.col("start_time_dt").dt.hour() < 12)
        & (pl.col("end_time_dt").dt.hour() < 12)
        & pl.col("end_station_name").is_not_null()
        & pl.col("start_station_name").is_not_null()
    )
    .group_by(
        [
            "end_date",
            "end_station_name",
            "start_station_name",
        ]
    )
    .agg(
        pl.col("trip_duration_num").mean().alias("avg_trip_duration"),
        pl.col("trip_id").count().alias("trip_count"),
    )
)

In [7]:
# trips count full day = 24,179,399
#trips count before 12 pm = 6,454,592
grouped_lazy.select(
    pl.col("trip_count").sum()
).collect()

trip_count
u32
6454592


In [ ]:
# save bike_riders_grouped_daily
grouped_lazy.sink_parquet(grouped_output_path)

In [11]:
grouped_pl = pl.scan_parquet(grouped_output_path)

grouped_pl.head(10).collect()

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count
date,str,str,f64,u32
2022-08-09,"""Spadina Ave / Adelaide St W""","""Lower Spadina Ave / Lake Shore…",416.0,1
2022-08-09,"""Bond St / Queen St E""","""Spadina Ave / Harbord St - SMA…",514.0,1
2022-08-09,"""Tecumseth St / Queen St W - SM…","""Ellis Ave / The Queensway""",1620.0,1
2022-08-09,"""University Ave / Armoury St""","""Front St E / Bayview Avenue""",1553.0,1
2022-08-09,"""King St W / Spadina Ave""","""Front St W / University Ave (2…",345.5,2
2022-08-09,"""Kilgour Rd / Rumsey Rd""","""Merton St / Mount Pleasant Rd""",800.0,1
2022-08-09,"""Wellington St W / Bay St""","""Berkeley St / Adelaide St E - …",493.0,1
2022-08-09,"""Vaughan Rd / Oakwood Ave""","""Symington Ave / Davenport Rd""",965.0,1
2022-08-09,"""Spadina Ave / Adelaide St W""","""457 King St. W. at Spadina""",138.0,1


In [12]:
grouped_pl.select(
    pl.len().alias("rows"),
    pl.col("avg_trip_duration").null_count().alias("null_avg_duration"),
    pl.col("trip_count").null_count().alias("total_trips"),
).collect()

rows,null_avg_duration,total_trips
u32,u32,u32
4312818,0,0


In [13]:
station_info_path = Path(
    "/home/najla/dev/najla-msc/data/raw/THATS/bike_riders/stations/station_information.json"
)

with open(station_info_path, "r", encoding="utf-8") as f:
    station_info = json.load(f)

stations_sub_pl = (
    pl.DataFrame(station_info["data"]["stations"])
    .select(
        pl.col("station_id").cast(pl.Int64),
        pl.col("name").cast(pl.Utf8).str.strip_chars(),
        pl.col("lat").cast(pl.Float64),
        pl.col("lon").cast(pl.Float64),
        pl.col("capacity").cast(pl.Int64, strict=False),
    )
)

stations_sub_pl.head()

station_id,name,lat,lon,capacity
i64,str,f64,f64,i64
7000,"""Fort York Blvd / Capreol Ct""",43.639832,-79.395954,47
7001,"""Wellesley Station Green P""",43.664964,-79.38355,23
7002,"""St. George St / Bloor St W""",43.667131,-79.399555,19
7003,"""Madison Ave / Bloor St W""",43.667018,-79.402796,15
7005,"""King St W / York St""",43.648001,-79.383177,23


In [14]:
stations_sub_pl.shape

(1046, 5)

In [15]:
extra_8_pl = pl.DataFrame(
    {
        "coords_name": [
            "2201 Dundas St W",
            "25 St. Mary St",
            "Dufferin St / King St W",
            "Ellesmere Rd / Scarborough Golf Club Rd",
            "King St W / Portland St",
            "Lawrence Ave E / Don Mills Rd",
            "Union Station (North) - SMART",
        ],
        "matched_end_station_name": [
            "2201 Dundas St W",
            "25 St. Mary St",
            "Dufferin St / King St W",
            "Ellesmere Rd / Scarborough Golf Club Rd",
            "King St W / Portland St - SMART",
            "Lawrence Ave E / Don Mills Rd",
            "Union Station (North)",
        ],
    }
).with_columns(
    pl.col("coords_name").cast(pl.Utf8).str.strip_chars(),
    pl.col("matched_end_station_name").cast(pl.Utf8).str.strip_chars(),
)

extra_8_pl

coords_name,matched_end_station_name
str,str
"""2201 Dundas St W""","""2201 Dundas St W"""
"""25 St. Mary St""","""25 St. Mary St"""
"""Dufferin St / King St W""","""Dufferin St / King St W"""
"""Ellesmere Rd / Scarborough Gol…","""Ellesmere Rd / Scarborough Gol…"
"""King St W / Portland St""","""King St W / Portland St - SMAR…"
"""Lawrence Ave E / Don Mills Rd""","""Lawrence Ave E / Don Mills Rd"""
"""Union Station (North) - SMART""","""Union Station (North)"""


In [16]:
stations_sub_updated_pl = (
    stations_sub_pl
    .join(
        extra_8_pl,
        how="left",
        left_on="name",
        right_on="coords_name",
    )
    .with_columns(
        pl.coalesce(
            pl.col("matched_end_station_name"),
            pl.col("name"),
        ).alias("name")
    )
    .drop("matched_end_station_name")
)

stations_sub_updated_pl.head()

station_id,name,lat,lon,capacity
i64,str,f64,f64,i64
7000,"""Fort York Blvd / Capreol Ct""",43.639832,-79.395954,47
7001,"""Wellesley Station Green P""",43.664964,-79.38355,23
7002,"""St. George St / Bloor St W""",43.667131,-79.399555,19
7003,"""Madison Ave / Bloor St W""",43.667018,-79.402796,15
7005,"""King St W / York St""",43.648001,-79.383177,23


In [17]:
stations_sub_updated_pl.filter(
    pl.col("name").is_in(extra_8_pl["matched_end_station_name"])
)

/tmp/ipykernel_23205/342211058.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  stations_sub_updated_pl.filter(


station_id,name,lat,lon,capacity
i64,str,f64,f64,i64
7229,"""2201 Dundas St W""",43.653427,-79.451279,11
7720,"""King St W / Portland St - SMAR…",43.644395,-79.400654,30
8001,"""Union Station (North)""",43.64565,-79.381163,24
8155,"""Ellesmere Rd / Scarborough Gol…",43.778815,-79.221741,15
8183,"""Dufferin St / King St W""",43.639171,-79.427268,23
8189,"""25 St. Mary St""",43.667569,-79.387494,15
8217,"""Lawrence Ave E / Don Mills Rd""",43.736889,-79.344278,14


In [18]:
grouped_clean_pl = grouped_pl.with_columns(
    pl.col("end_station_name").cast(pl.Utf8).str.strip_chars(),
    pl.col("start_station_name").cast(pl.Utf8).str.strip_chars(),
)

In [19]:
end_station_info_pl = stations_sub_updated_pl.select(
    pl.col("name"),
    pl.col("station_id").alias("end_station_id"),
    pl.col("lat").alias("lat_end"),
    pl.col("lon").alias("lon_end"),
    pl.col("capacity").alias("capacity_end"),
)

grouped_with_end_pl = (
    grouped_clean_pl
    .join(
        end_station_info_pl.lazy(),
        how="left",
        left_on="end_station_name",
        right_on="name",
    )
)

grouped_with_end_pl.filter(pl.col("end_station_id").is_not_null()).head(10).collect()

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end
date,str,str,f64,u32,i64,f64,f64,i64
2022-08-09,"""Spadina Ave / Adelaide St W""","""Lower Spadina Ave / Lake Shore…",416.0,1,7260,43.647202,-79.395339,15
2022-08-09,"""Bond St / Queen St E""","""Spadina Ave / Harbord St - SMA…",514.0,1,7045,43.653236,-79.376716,37
2022-08-09,"""King St W / Spadina Ave""","""Front St W / University Ave (2…",345.5,2,7010,43.645142,-79.395104,19
2022-08-09,"""Kilgour Rd / Rumsey Rd""","""Merton St / Mount Pleasant Rd""",800.0,1,7663,43.718039,-79.371914,17
2022-08-09,"""Wellington St W / Bay St""","""Berkeley St / Adelaide St E - …",493.0,1,7052,43.647259,-79.379878,55
2022-08-09,"""Vaughan Rd / Oakwood Ave""","""Symington Ave / Davenport Rd""",965.0,1,7618,43.692682,-79.441042,11
2022-08-09,"""Spadina Ave / Adelaide St W""","""457 King St. W. at Spadina""",138.0,1,7260,43.647202,-79.395339,15
2022-08-09,"""Frederick St / King St E""","""Sherbourne St N / Elm Ave""",786.0,1,7432,43.651118,-79.369411,47
2022-08-09,"""Front St W / Blue Jays Way""","""Summerhill Station""",1513.0,1,7059,43.643473,-79.390477,15


In [21]:
grouped_with_end_pl.count().collect()

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end
u32,u32,u32,u32,u32,u32,u32,u32,u32
4312818,4312818,4312818,4312818,4312818,3665275,3665275,3665275,3665275


In [22]:
grouped_with_end_filtered_pl = grouped_with_end_pl.filter(pl.col("end_station_id").is_not_null())
grouped_with_end_filtered_pl.count().collect()

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end
u32,u32,u32,u32,u32,u32,u32,u32,u32
3665275,3665275,3665275,3665275,3665275,3665275,3665275,3665275,3665275


In [23]:
start_station_info_pl = stations_sub_updated_pl.select(
    pl.col("name"),
    pl.col("station_id").alias("start_station_id"),
    pl.col("lat").alias("lat_start"),
    pl.col("lon").alias("lon_start"),
    pl.col("capacity").alias("capacity_start"),
)

grouped_with_start_pl = (
    grouped_with_end_filtered_pl
    .join(
        start_station_info_pl.lazy(),
        how="left",
        left_on="start_station_name",
        right_on="name",
    )
)

grouped_with_start_pl.head(10).collect()

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end,start_station_id,lat_start,lon_start,capacity_start
date,str,str,f64,u32,i64,f64,f64,i64,i64,f64,f64,i64
2022-08-09,"""Spadina Ave / Adelaide St W""","""Lower Spadina Ave / Lake Shore…",416.0,1,7260,43.647202,-79.395339,15,null,null,null,null
2022-08-09,"""Bond St / Queen St E""","""Spadina Ave / Harbord St - SMA…",514.0,1,7045,43.653236,-79.376716,37,7285,43.662923,-79.401845,36
2022-08-09,"""King St W / Spadina Ave""","""Front St W / University Ave (2…",345.5,2,7010,43.645142,-79.395104,19,7321,43.645297,-79.382791,19
2022-08-09,"""Kilgour Rd / Rumsey Rd""","""Merton St / Mount Pleasant Rd""",800.0,1,7663,43.718039,-79.371914,17,null,null,null,null
2022-08-09,"""Wellington St W / Bay St""","""Berkeley St / Adelaide St E - …",493.0,1,7052,43.647259,-79.379878,55,7506,43.653269,-79.36482,12
2022-08-09,"""Vaughan Rd / Oakwood Ave""","""Symington Ave / Davenport Rd""",965.0,1,7618,43.692682,-79.441042,11,7529,43.67044,-79.453285,15
2022-08-09,"""Spadina Ave / Adelaide St W""","""457 King St. W. at Spadina""",138.0,1,7260,43.647202,-79.395339,15,null,null,null,null
2022-08-09,"""Frederick St / King St E""","""Sherbourne St N / Elm Ave""",786.0,1,7432,43.651118,-79.369411,47,7530,43.675273,-79.377846,22
2022-08-09,"""Front St W / Blue Jays Way""","""Summerhill Station""",1513.0,1,7059,43.643473,-79.390477,15,null,null,null,null


In [24]:
grouped_with_start_pl.count().collect()

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end,start_station_id,lat_start,lon_start,capacity_start
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
3665275,3665275,3665275,3665275,3665275,3665275,3665275,3665275,3665275,3215817,3215817,3215817,3215817


In [25]:
grouped_with_filtered = grouped_with_start_pl.filter(pl.col("start_station_id").is_not_null())
grouped_with_filtered.count().collect()

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end,start_station_id,lat_start,lon_start,capacity_start
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
3215817,3215817,3215817,3215817,3215817,3215817,3215817,3215817,3215817,3215817,3215817,3215817,3215817


In [ ]:
"""bike_trips_stations_info = Path(
    "/home/najla/dev/najla-msc/data/processed/THATS/bike_trips_stations_info.parquet"
)"""
bike_trips_stations_info = Path(
    "/home/najla/dev/najla-msc/bikeshare/data/processed/bike_ridership/bike_trips_stations_info_5yrs_12pm.parquet"
)
# grouped_with_filtered.sink_parquet(bike_trips_stations_info)

Explore year

In [8]:
from datetime import datetime
import polars as pl
(
    bike_riders_lazy
    .with_columns(
        pl.col("end_time")
        .str.to_datetime(strict=False)
        .alias("end_time_dt")
    )
    .filter(pl.col("end_time_dt") >= datetime(2025, 1, 1))
    .limit(20)
    .collect()
)

trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,source_file,bike_model,end_time_dt
i64,str,f64,str,str,f64,str,str,i64,str,str,str,datetime[μs]


Produce daily station before 12 pm 

In [4]:
from datetime import datetime
import polars as pl

(
    bike_riders_lazy
    .with_columns(
        pl.coalesce(
            pl.col("start_time").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False),
            pl.col("start_time").str.to_datetime(format="%m/%d/%Y %H:%M", strict=False),
        ).alias("start_time_dt")
    )
    .filter(
        (pl.col("start_time_dt") >= datetime(2026, 1, 1)) &
        (pl.col("start_time_dt") < datetime(2027, 1, 1))
    )
    .limit(5)
    .collect()
)

trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,source_file,bike_model,start_time_dt
i64,str,f64,str,str,f64,str,str,i64,str,str,str,datetime[μs]
43682415,"""248""",7719.0,"""2026-01-01 00:00:26""","""Wolseley St / Augusta Ave""",7037.0,"""2026-01-01 00:04:34""","""Bathurst St / Dundas St W""",122,"""Member""","""bikeshare-ridership-2026.csv""","""ICONIC""",2026-01-01 00:00:26
43682417,"""295""",8001.0,"""2026-01-01 00:02:05""","""Union Station (North)""",7076.0,"""2026-01-01 00:07:00""","""York St / Queens Quay W""",4900,"""Member""","""bikeshare-ridership-2026.csv""","""ICONIC""",2026-01-01 00:02:05
43682418,"""2701""",7172.0,"""2026-01-01 00:04:15""","""Strachan Ave / Princes' Blvd""",7100.0,"""2026-01-01 00:49:16""","""Dundas St E / Regent Park Blvd""",1833,"""Member""","""bikeshare-ridership-2026.csv""","""ICONIC""",2026-01-01 00:04:15
43682419,"""581""",7259.0,"""2026-01-01 00:04:29""","""Lower Spadina Ave / Lake Shore…",7076.0,"""2026-01-01 00:14:10""","""York St / Queens Quay W""",5078,"""Casual""","""bikeshare-ridership-2026.csv""","""ICONIC""",2026-01-01 00:04:29
43682420,"""600""",7259.0,"""2026-01-01 00:04:53""","""Lower Spadina Ave / Lake Shore…",7076.0,"""2026-01-01 00:14:53""","""York St / Queens Quay W""",3951,"""Member""","""bikeshare-ridership-2026.csv""","""ICONIC""",2026-01-01 00:04:53


In [15]:
from datetime import date
import polars as pl

(
    grouped_lazy #bike_riders_grouped_daily
    .filter(
        (pl.col("end_date") >= date(2025, 1, 1)) &
        (pl.col("end_date") < date(2026, 1, 1))
    )
    .limit(5)
    .collect()
)

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count
date,str,str,f64,u32
2025-11-15,"""Mill St / Cherry St""","""St. Andrew's Playground Park""",1334.0,1
2025-11-07,"""College St / Markham St""","""Yonge St / Alexander St - SMAR…",916.0,1
2025-11-26,"""Adelaide St W / Yonge St""","""Parliament St / Aberdeen Ave""",656.0,1
2025-11-24,"""Toronto Eaton Centre (Yonge St…","""Palmerston Ave / Dundas St W""",641.0,1
2025-09-11,"""The Queensway / High St""","""The Queensway / High St""",1747.0,2
